<a href="https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- One row = one content item (content_hash_id) for one client (client_hash_id) on one day (report_date) — grain is daily x client x content, from fact_content_daily_performance.
- Table(s): dim_content (metadata/joins) + fact_content_daily_performance (daily signals).
- Time window: developing on month=2026-03 (mid-panel, avoids the sealed _sample month).
- Label/proxy: whether a page's impressions decline over a forward 30-day window relative to a trailing 90-day baseline — a stronger version of the starter's trend_direction == "down" proxy.
- One thing excluded: any FlyRank product decision flag (health_score, priority_score, action_type) — these are rule outputs, not observable signals, and including them would make the model just copy the existing rule (circular result).

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` | Feature | observable daily signal, known before decision point |
| `gsc_clicks` | Feature | observable daily signal, known before decision point |
| `gsc_avg_position` | Feature | observable daily signal, known before decision point |
| `ga4_engaged_sessions` / `ga4_sessions` (→ engagement_rate) | Feature | derived from same-day GA4 signals, known before decision point |
| `scroll_events` / `ga4_pageviews` (→ scroll_rate) | Feature | derived from same-day signals, known before decision point |
| Forward 30-day impression trend | Label | the outcome we're trying to predict/rank |
| `content_hash_id`, `client_hash_id` | Context | join keys only, carry no signal themselves |
| `report_date`, `month` | Context | partition/join keys, not model features |
| `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | Context | availability flags — used to filter/validate, not as features |
| Any FlyRank product score (`health_score`, `priority_score`, `action_type` — not present in this table, confirmed via DESCRIBE) | Excluded | product decision outputs are deliberately not shipped in this data; would cause a circular result if used |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

With the leaked feature, AUC was ~X (too good to be true). Removed it — honest AUC is Y.

In [ ]:
!pip install -q huggingface_hub duckdb
import duckdb, os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

base = "hf://datasets/FlyRank/internship-warehouse"

# Query 1 — grain check: is one row really client+content+day?
con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""").show()  # should return 0 rows if grain holds

# Query 2 — row count & date span for the month
con.sql(f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
       COUNT(DISTINCT content_hash_id) AS n_content
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

# Query 3 — availability, filtered with IS TRUE
con.sql(f"""
SELECT COUNT(*) AS total,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   n   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

┌─────────┬────────────┬────────────┬───────────┐
│ n_rows  │   min_d    │   max_d    │ n_content │
│  int64  │    date    │    date    │   int64   │
├─────────┼────────────┼────────────┼───────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │    331437 │
└─────────┴────────────┴────────────┴───────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────────────┐
│  total  │ ga4_available_rows │
│  int64  │       int64        │
├─────────┼────────────────────┤
│ 9841378 │             413966 │
└─────────┴────────────────────┘



In [ ]:
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
features = con.sql(f"""
SELECT
  content_hash_id,
  AVG(gsc_impressions) AS avg_impressions,                              -- available: known daily, before decision point
  AVG(gsc_clicks) AS avg_clicks,                                         -- available: known daily
  AVG(gsc_avg_position) AS avg_position,                                 -- available: known daily
  SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,  -- available: known daily, derived from same-day GA4 signals
  SUM(scroll_events) * 1.0 / NULLIF(SUM(ga4_pageviews), 0) AS scroll_rate            -- available: known daily, derived from same-day pageviews
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The clearest limit in this slice: GA4 availability is sparse relative to search data. Of 9,841,378 daily rows in `month=2026-03`, only 413,966 (~4.2%) have `ga4_data_available IS TRUE`. This means most rows in this month carry search-console signal only (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) with no engagement or session data behind them — so any feature built from `ga4_*` or `sessions_*` columns (like `engagement_rate` or `scroll_rate`) is missing or zero for the large majority of the panel, not because engagement was actually zero, but because tracking wasn't active yet for that client at that point.

This is a coverage limit, not a correctness limit: where GA4 data exists, it's real. But it means engagement-based features are only reliable for the minority of rows where `ga4_data_available IS TRUE`, and any model or ranking that leans heavily on those features will effectively be scored on a smaller, non-random subset of content — clients who onboarded GA4 tracking earlier. This data cannot tell us anything about engagement behavior for the ~96% of rows where GA4 wasn't yet tracking, and treating a missing/zero engagement value as "no engagement" instead of "no data" would bias any downstream ranking or model toward clients with longer GA4 history.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.